# 2b. Baseline Training - RandomForest

Train a strong baseline RandomForest on the processed 30-feature split and save artifacts for evaluation notebook 3b.

In [1]:
import os, numpy as np, pandas as pd, joblib, pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

base_dir = os.path.join('..')
splits_dir = os.path.join(base_dir, 'data', 'splits')
models_dir = os.path.join(base_dir, 'models')
SEED = 42

In [2]:
train_df = pd.read_csv(os.path.join(splits_dir, 'train.csv'))
val_df = pd.read_csv(os.path.join(splits_dir, 'val.csv'))
test_df = pd.read_csv(os.path.join(splits_dir, 'test.csv'))

selected_features = joblib.load(os.path.join(models_dir, 'selected_features.pkl'))
le = joblib.load(os.path.join(models_dir, 'label_encoder.pkl'))
class_names = le.classes_.tolist()

print('Train/Val/Test:', train_df.shape, val_df.shape, test_df.shape)
print('Num selected features:', len(selected_features))
print('Classes:', class_names)

Train/Val/Test: (448000, 31) (35069, 31) (82332, 31)
Num selected features: 30
Classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


In [3]:
X_train = train_df[selected_features].values
y_train = train_df['label'].values.astype(int)

X_val = val_df[selected_features].values
y_val = val_df['label'].values.astype(int)

X_test = test_df[selected_features].values
y_test = test_df['label'].values.astype(int)

print('X_train:', X_train.shape, 'X_val:', X_val.shape, 'X_test:', X_test.shape)

X_train: (448000, 30) X_val: (35069, 30) X_test: (82332, 30)


In [4]:
rf_params = dict(
    n_estimators=1000,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=SEED
)

rf = RandomForestClassifier(**rf_params)
rf.fit(X_train, y_train)

print('Baseline RandomForest trained.')

Baseline RandomForest trained.


In [5]:
def metrics_dict(y_true, y_pred):
    return {
        'acc': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

val_pred = rf.predict(X_val)
val_probs = rf.predict_proba(X_val)
val_metrics = metrics_dict(y_val, val_pred)

print('Validation metrics:', val_metrics)
print(classification_report(y_val, val_pred, target_names=class_names, zero_division=0))

Validation metrics: {'acc': 0.774273574952237, 'precision_macro': 0.6100081865462708, 'recall_macro': 0.5996688538533206, 'f1_macro': 0.5920185998008027, 'f1_weighted': 0.8053564557153188}
                precision    recall  f1-score   support

      Analysis       0.03      0.14      0.05       400
      Backdoor       0.02      0.16      0.04       349
           DoS       0.30      0.24      0.27      2453
      Exploits       0.83      0.55      0.67      6679
       Fuzzers       0.75      0.80      0.77      3637
       Generic       1.00      0.98      0.99      8000
        Normal       0.96      0.91      0.94     11200
Reconnaissance       0.89      0.76      0.82      2098
     Shellcode       0.69      0.76      0.72       227
         Worms       0.62      0.69      0.65        26

      accuracy                           0.77     35069
     macro avg       0.61      0.60      0.59     35069
  weighted avg       0.85      0.77      0.81     35069



In [6]:
test_pred = rf.predict(X_test)
test_probs = rf.predict_proba(X_test)
test_metrics = metrics_dict(y_test, test_pred)

history = {
    'model': 'baseline_random_forest',
    'rf_params': rf_params,
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'selected_features': selected_features
}

joblib.dump(rf, os.path.join(models_dir, 'baseline_rf_model.pkl'))
np.save(os.path.join(models_dir, 'baseline_rf_preds.npy'), test_pred)
np.save(os.path.join(models_dir, 'baseline_rf_true.npy'), y_test)
np.save(os.path.join(models_dir, 'baseline_rf_probs.npy'), test_probs)

with open(os.path.join(models_dir, 'history_baseline_rf.pkl'), 'wb') as f:
    pickle.dump(history, f)

print('Test metrics:', test_metrics)
print(classification_report(y_test, test_pred, target_names=class_names, zero_division=0))
print('Saved: baseline_rf_model.pkl, baseline_rf_preds.npy, baseline_rf_true.npy, baseline_rf_probs.npy, history_baseline_rf.pkl')

Test metrics: {'acc': 0.730141378807754, 'precision_macro': 0.5301826575713338, 'recall_macro': 0.5787627658268796, 'f1_macro': 0.5152093985604529, 'f1_weighted': 0.7703608876149384}
                precision    recall  f1-score   support

      Analysis       0.03      0.07      0.04       677
      Backdoor       0.05      0.45      0.09       583
           DoS       0.36      0.17      0.23      4089
      Exploits       0.75      0.66      0.70     11132
       Fuzzers       0.28      0.62      0.38      6062
       Generic       1.00      0.97      0.98     18871
        Normal       0.97      0.72      0.82     37000
Reconnaissance       0.90      0.81      0.86      3496
     Shellcode       0.31      0.74      0.43       378
         Worms       0.66      0.57      0.61        44

      accuracy                           0.73     82332
     macro avg       0.53      0.58      0.52     82332
  weighted avg       0.85      0.73      0.77     82332

Saved: baseline_rf_model.pkl, 